# Stage 5: LSTM

Results of `python -m src.train_lstm` (15-min and 1-h horizons; the day-ahead LSTM is a stretch goal
and was cut, see the `PLAN.md` Change log).

**Input:** the last rows of the same feature table the GBMs use (16 rows = 4 h for 15 min, 24 rows = 1 day
for 1 h), as a sequence. Every value in the window is known when the forecast is issued.

**Protocol:** the same as for the GBMs. Two hidden sizes are tried on train with early stopping on
validation; the better one is refit on train + validation for the same number of epochs; the test year
is scored once.

In [ ]:
import pandas as pd
from IPython.display import Image

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
TABLES = "../results/tables/"
FIGURES = "../results/figures/"

## 1. Training curves

In [ ]:
Image(FIGURES + "05_lstm_curves.png")

In [ ]:
training = pd.read_csv(TABLES + "05_lstm_training.csv")
search = training[training["run"].str.startswith("train")]
search.groupby(["horizon", "hidden"])["val_rmse"].agg(["min", "count"]).round(5)

## 2. LSTM next to every other model (test, daytime)

In [ ]:
names = ["03_baselines.csv", "04_gbm_results.csv", "05_lstm_results.csv"]
results = pd.concat([pd.read_csv(TABLES + n) for n in names])
test = results[(results["split"] == "test") & (results["hours"] == "daytime")]
test = test[test["horizon"] != "dayahead"]

In [ ]:
columns = ["horizon", "model", "nmae_pct", "nmae_pct_low", "nmae_pct_high", "skill", "skill_low", "skill_high"]
test[columns].sort_values(["horizon", "nmae_pct"]).round(3)

In [ ]:
Image(FIGURES + "05_models_skill.png")

## 3. Is the deep model worth its cost?
Training time of the final fit (seconds, this laptop CPU). The latency and size comparison is in Stage 6.

In [ ]:
lstm = pd.read_csv(TABLES + "05_lstm_results.csv")
gbm = pd.read_csv(TABLES + "04_gbm_results.csv")
cost = pd.concat([lstm, gbm])
cost = cost[(cost["split"] == "test") & (cost["hours"] == "daytime") & (cost["horizon"] != "dayahead")]
cost[["horizon", "model", "fit_seconds", "nmae_pct", "skill"]].sort_values(["horizon", "nmae_pct"]).round(3)